# Lab 1 – Crop và Resize ảnh

**Người thực hiện:** TV5  
**Branch:** `feature/lab01-crop-resize`  
**Source:** `lab_1/src/transform.py`

## Mục tiêu
Thực hiện crop và resize, hiểu đúng tọa độ ảnh và vấn đề tỷ lệ khung hình (aspect ratio).

## 1. Import và đọc ảnh

In [ ]:
import sys
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(str(Path("..") / "src"))
import transform as tf

from image_io import find_input_image
INPUT = find_input_image("06_animal_lion.jpg")   # ảnh trong data/input (tự chọn ảnh khác nếu không thấy)
OUTPUT_DIR = Path("../data/output")
FIG_DIR = Path("../result/figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

image = cv2.imread(str(INPUT))
if image is None:
    raise FileNotFoundError(f"Không đọc được ảnh: {INPUT}")
H, W = image.shape[:2]
print(f"Ảnh gốc: width={W}, height={H}, tỷ lệ W/H = {tf.aspect_ratio(image):.3f}")

def rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

## 2. Tọa độ ảnh
- Gốc `(0, 0)` ở **góc trên-trái**; `x` là **cột** (tăng sang phải), `y` là **hàng** (tăng xuống dưới).
- Ảnh là mảng NumPy `(H, W, C)` nên slicing viết **hàng trước, cột sau**: `image[y1:y2, x1:x2]`.
- `y1:y2` và `x1:x2` theo kiểu nửa mở: lấy từ `y1` đến `y2 - 1`, nên kích thước crop là `(y2 - y1) × (x2 - x1)`.
- Hàm vẽ của OpenCV lại dùng `(x, y)`, đây là chỗ rất dễ nhầm.

## 3. Crop một vùng có ý nghĩa
Cắt hai vùng: **vùng trung tâm** (chủ thể chính) và **góc trên-phải**. Tọa độ tính theo tỷ lệ kích thước ảnh nên dùng được cho mọi ảnh; có thể chỉnh các tỷ lệ này để cắt đúng vùng mong muốn.

In [ ]:
# Vùng 1: trung tâm (25%-75% chiều rộng, 20%-80% chiều cao)
x1, y1, x2, y2 = int(0.25 * W), int(0.20 * H), int(0.75 * W), int(0.80 * H)
crop_a = tf.crop_image(image, x1, y1, x2, y2)

# Vùng 2: góc trên-phải (60%-100% chiều rộng, 0%-40% chiều cao)
cx1, cy1, cx2, cy2 = int(0.60 * W), 0, W, int(0.40 * H)
crop_b = tf.crop_image(image, cx1, cy1, cx2, cy2)

print("Crop trung tâm :", crop_a.shape,  f"= (y2-y1, x2-x1, 3) = ({y2-y1}, {x2-x1}, 3)")
print("Crop trên-phải :", crop_b.shape)

# Vẽ khung vùng crop lên ảnh gốc để kiểm tra trực quan
marked = image.copy()
cv2.rectangle(marked, (x1, y1), (x2, y2), (0, 0, 255), 3)      # OpenCV: (x, y)
cv2.rectangle(marked, (cx1, cy1), (cx2, cy2), (255, 0, 0), 3)

plt.figure(figsize=(15, 4))
plt.subplot(1, 3, 1); plt.imshow(rgb(marked)); plt.title(f"Ảnh gốc {W}x{H} và vùng crop"); plt.axis("off")
plt.subplot(1, 3, 2); plt.imshow(rgb(crop_a));  plt.title(f"Crop trung tâm {crop_a.shape[1]}x{crop_a.shape[0]}"); plt.axis("off")
plt.subplot(1, 3, 3); plt.imshow(rgb(crop_b)); plt.title(f"Crop trên-phải {crop_b.shape[1]}x{crop_b.shape[0]}"); plt.axis("off")
plt.savefig(FIG_DIR / "crop_regions.png", dpi=150, bbox_inches="tight")
plt.show()

### Kiểm tra lỗi: tọa độ ngoài ảnh

In [ ]:
try:
    tf.crop_image(image, 0, 0, W + 100, 100)     # x2 = W + 100 > width
except ValueError as e:
    print("Bắt được lỗi:", e)

**Giải thích tọa độ:** crop trung tâm lấy từ cột `x1` đến `x2 - 1` và hàng `y1` đến `y2 - 1`, nên mảng kết quả có shape `(y2 - y1, x2 - x1, 3)`. Nếu nhầm thứ tự thành `image[x1:x2, y1:y2]` sẽ lấy sai vùng, hoặc ra mảng rỗng khi `x` vượt chiều cao ảnh.

In [ ]:
cv2.imwrite(str(OUTPUT_DIR / f"{INPUT.stem}_crop_a.jpg"), crop_a)
cv2.imwrite(str(OUTPUT_DIR / f"{INPUT.stem}_crop_b.jpg"), crop_b)
print("Đã lưu ảnh crop")

## 4. Resize theo tỷ lệ (50% và 25%)
`cv2.resize(image, None, fx=0.5, fy=0.5)`: khi truyền `fx, fy` thì tham số kích thước đích để `None`.

In [ ]:
r50 = tf.resize_by_scale(image, 0.5)
r25 = tf.resize_by_scale(image, 0.25)

for name, im in [("Gốc", image), ("50%", r50), ("25%", r25)]:
    print(f"{name:5s}: shape={im.shape}  (W={im.shape[1]}, H={im.shape[0]})  W/H={tf.aspect_ratio(im):.3f}")

plt.figure(figsize=(15, 4))
for i, (im, t) in enumerate([(image, "Gốc"), (r50, "Resize 50%"), (r25, "Resize 25%")], 1):
    plt.subplot(1, 3, i); plt.imshow(rgb(im)); plt.title(f"{t} - {im.shape[1]}x{im.shape[0]}")
plt.savefig(FIG_DIR / "resize_scale.png", dpi=150, bbox_inches="tight")
plt.show()

> Matplotlib tự co ảnh vừa khung nên 3 ảnh trông có vẻ cùng cỡ; hãy nhìn **trục tọa độ** (số pixel) để thấy kích thước thật. Dòng `print` ở trên cho kích thước chính xác. Tỷ lệ W/H không đổi vì `fx = fy`.

In [ ]:
cv2.imwrite(str(OUTPUT_DIR / f"{INPUT.stem}_resize_50.jpg"), r50)
cv2.imwrite(str(OUTPUT_DIR / f"{INPUT.stem}_resize_25.jpg"), r25)

## 5. Resize về kích thước cố định
`cv2.resize(image, (width, height))` - lưu ý OpenCV nhận `(width, height)`, ngược thứ tự với `shape`.

In [ ]:
fixed_same_ratio = tf.resize_fixed(image, W // 2, H // 2)          # đúng một nửa -> giữ tỷ lệ
fixed_43         = tf.resize_fixed(image, W // 2, (W // 2) * 3 // 4)  # tỷ lệ 4:3 -> khác ảnh gốc, bị méo
fixed_square     = tf.resize_fixed(image, 300, 300)                  # ép vuông -> méo nhiều
keep_w300        = tf.resize_keep_ratio(image, width=300)            # chỉ cho width, height tự tính

for name, im in [("W/2 x H/2 (giữ tỷ lệ)", fixed_same_ratio), ("Tỷ lệ 4:3", fixed_43), ("300x300 (vuông)", fixed_square), ("width=300 giữ tỷ lệ", keep_w300)]:
    print(f"{name:24s} -> shape={im.shape}, W/H={tf.aspect_ratio(im):.3f}")
print(f"{'Ảnh gốc':24s} -> shape={image.shape}, W/H={tf.aspect_ratio(image):.3f}")

## 6. So sánh giữ tỷ lệ và thay đổi aspect ratio
Hình tròn là hình dễ nhìn nhất để phát hiện méo: nếu tỷ lệ bị đổi, hình tròn thành hình elip.

In [ ]:
# Ảnh thử có hình tròn và hình vuông chuẩn
test = np.full((400, 600, 3), 255, np.uint8)
cv2.circle(test, (300, 200), 120, (0, 0, 255), -1)
cv2.rectangle(test, (20, 20), (120, 120), (255, 0, 0), -1)

ok  = tf.resize_keep_ratio(test, width=300)     # 300x200, giữ tỷ lệ
bad = tf.resize_fixed(test, 300, 300)           # 300x300, bị ép vuông
bad2 = tf.resize_fixed(test, 600, 200)          # bị bẹp

plt.figure(figsize=(15, 4))
for i, (im, t) in enumerate([(test, "Gốc 600x400"), (ok, "Giữ tỷ lệ 300x200"), (bad, "Ép 300x300 (méo)"), (bad2, "Ép 600x200 (méo)")], 1):
    plt.subplot(1, 4, i); plt.imshow(rgb(im)); plt.title(t)
plt.savefig(FIG_DIR / "resize_aspect_ratio.png", dpi=150, bbox_inches="tight")
plt.show()

**Giải thích hiện tượng méo:** khi tỷ lệ `width/height` mới khác tỷ lệ gốc, trục ngang và trục dọc bị co/giãn với hệ số khác nhau. Hình tròn thành elip, hình vuông thành chữ nhật. Muốn giữ đúng tỷ lệ thì dùng cùng một hệ số cho hai chiều (`fx = fy`) hoặc chỉ cho một cạnh rồi tính cạnh còn lại (`resize_keep_ratio`). Nếu bắt buộc ra kích thước cố định, có thể resize giữ tỷ lệ rồi thêm viền (padding) hoặc crop phần thừa.

## 7. Chất lượng ảnh khi resize: các phương pháp nội suy
Resize cần tạo pixel mới nên phải **nội suy**. Phương pháp ảnh hưởng chất lượng.

In [ ]:
small = tf.resize_by_scale(crop_a, 0.25)   # thu nhỏ rồi phóng lại để so sánh chi tiết bị mất
methods = {"INTER_NEAREST": cv2.INTER_NEAREST, "INTER_LINEAR": cv2.INTER_LINEAR, "INTER_CUBIC": cv2.INTER_CUBIC, "INTER_AREA": cv2.INTER_AREA}

plt.figure(figsize=(16, 4))
plt.subplot(1, 5, 1); plt.imshow(rgb(small)); plt.title(f"Ảnh nhỏ {small.shape[1]}x{small.shape[0]}")
for i, (name, m) in enumerate(methods.items(), 2):
    up = cv2.resize(small, (crop_a.shape[1] * 2, crop_a.shape[0] * 2), interpolation=m)
    plt.subplot(1, 5, i); plt.imshow(rgb(up)); plt.title(f"Phóng to x2: {name}", fontsize=9)
plt.savefig(FIG_DIR / "resize_interpolation.png", dpi=150, bbox_inches="tight")
plt.show()

- `INTER_NEAREST`: nhanh nhất nhưng bị răng cưa, vỡ khối.
- `INTER_LINEAR`: mặc định, cân bằng tốc độ và độ mượt.
- `INTER_CUBIC`: mượt và sắc hơn khi **phóng to**, chậm hơn.
- `INTER_AREA`: phù hợp nhất khi **thu nhỏ** vì giảm hiện tượng răng cưa.

## 8. Nhận xét
- **Crop** thay đổi *vùng nội dung* (giữ nguyên độ phân giải của vùng đó); **resize** thay đổi *kích thước toàn bộ ảnh* (nội dung giữ nguyên, số pixel đổi).
- Crop bằng slicing nhanh nhưng trả về *view* của ảnh gốc; sửa crop có thể làm đổi ảnh gốc, nên dùng `.copy()` như trong `crop_image`.
- Thu nhỏ làm mất chi tiết vĩnh viễn: phóng lại ảnh nhỏ không lấy lại được độ nét ban đầu.
- Aspect ratio là nguyên nhân chính gây méo ảnh; cần giữ `fx = fy` hoặc dùng `resize_keep_ratio`.
- Khi gọi `cv2.resize` với kích thước cố định phải truyền `(width, height)`, ngược thứ tự với `image.shape`.